In [ ]:
import dashscope
print(dashscope.__version__)

In [ ]:
# 导入库
import dashscope
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# 设置你的api_key

# 读取数据集
df = pd.read_csv("pima-indians-diabetes.csv")
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# 划分数据集，训练逻辑回归模型
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 随便取一位测试集患者的数据
patient = X_test.iloc[0].to_dict()
risk_prob = model.predict_proba(X_test.iloc[0:1])[0,1]

print("患者指标：", patient)
print(f"模型预测糖尿病患病概率：{risk_prob:.2%}")

# 调用Qwen‑Max给出专业解读
prompt = f"""
你是一名内分泌临床医师。根据下面患者体检指标和模型给出的患病风险，给一段通俗易懂的解读和生活干预建议，不要过度诊断，提示仅供参考，不能替代临床就诊。
患者指标：{patient}
模型预测患病概率：{risk_prob:.2%}
"""

resp = dashscope.Generation.call(
    model="qwen-max",
    messages=[{"role":"user","content": prompt}]
)
if resp.status_code == 200:
    print("\n===== 模型解读 =====")
    print(resp.output.text)
else:
    print("调用失败：",resp.message)
